<a href="https://colab.research.google.com/github/jl17243-commits/Factor-Investment/blob/main/Factor_investment_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

因子风险暴露估计


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

模拟数据（需替换）

In [ ]:
ASSETS = [
    "USRiskFreeRate", "UST_Index", "TIPS", "US_Corp_Credit_IG", "US_Corp_Credit_HY","EM_IG", "EM_Broad",
    "US_MBS_Agency", "US_CMBS", "US_ABS",
    "US_Lev_Loans", "US_Public_Equity", "US_Private_Equity","US_Real_Estate",
    "Global_Hedge_Funds", "AbsRtn_Funds","US_Private_Equity_Unsmthd"
    , "US_Real_Estate_Unsmthd"
]
FACTOR_NAMES = ["Equities", "US Rates", "Credit", "StructCredit", "Risk Free Rate"]

raw_factors =pd.read_excel("/content/data.xlsx",sheet_name="Sheet1",index_col=0,parse_dates=True)

df=pd.read_excel("/content/data.xlsx",sheet_name="Sheet2",index_col=0,parse_dates=True)


In [ ]:
def orthogonalize_factors(factors):
  orth=pd.DataFrame(index=factors.index,columns=factors.columns,dtype=float)
  orth.iloc[:,0]=factors.iloc[:,0]

  for k in range(1,factors.shape[1]):
    y=factors.iloc[:,k].values
    X=add_constant(orth.iloc[:,:k].values)
    res=OLS(y,X).fit()
    orth.iloc[:,k]=res.resid

  return orth

orth_factors=orthogonalize_factors(raw_factors)
print(orth_factors.corr().round(4))

In [ ]:
def newey_west_lags(T,freq="monthly"):
  return int(np.floor(4*(T/100)**(2/9)))

def run_ols_nw(asset_returns,orth_factors,nlags):
  y=asset_returns.values
  X=add_constant(orth_factors.values)
  model=OLS(y,X).fit()

  cov_nw=cov_hac(model,nlags=nlags)
  se_nw=np.sqrt(np.diag(cov_nw))
  t_nw=model.params/se_nw
  p_nw=2*stats.t.sf(np.abs(t_nw),df=model.df_resid)

  return {
      "alpha":   model.params[0],
      "betas":   model.params[1:],
      "resid_var": model.mse_resid,
      "t_stats":  t_nw[1:],
      "p_vals":  p_nw[1:],
      "r2":    model.rsquared,
      "residuals": model.resid,
  }

In [ ]:
T=len(df)
nlags=newey_west_lags(T)
print(nlags)

In [ ]:
betas_rows,alpha_rows,resid_var_rows=[],[],[]
tstat_rows,pval_rows,r2_rows=[],[],[]
all_residuals={}

for asset in df.columns:
  res=run_ols_nw(df[asset],orth_factors,nlags)
  betas_rows.append(res["betas"])
  alpha_rows.append(res["alpha"])
  resid_var_rows.append(res["resid_var"])
  tstat_rows.append(res["t_stats"])
  pval_rows.append(res["p_vals"])
  r2_rows.append(res["r2"])
  all_residuals[asset]=res["residuals"]

betas=pd.DataFrame(betas_rows,index=df.columns,columns=FACTOR_NAMES)
alphas=pd.Series(alpha_rows,index=df.columns,name="alpha")
resid_var=pd.Series(resid_var_rows,index=df.columns,name="resid_var")
tstat_df=pd.DataFrame(tstat_rows,index=df.columns,columns=FACTOR_NAMES)
pval_df=pd.DataFrame(pval_rows,index=df.columns,columns=FACTOR_NAMES)
r2=pd.Series(r2_rows,index=df.columns,name="R2")

print("\n── Beta matrix (18 × 5) ──")
print(betas.round(4))
print("\n── R² per asset ──")
print(r2.round(4))
print("\n── Newey-West t-statistics ──")
print(tstat_df.round(2))


In [ ]:
def sig_stars(p):
  if p < 0.01:  return "***"
  if p < 0.05:  return "**"
  if p < 0.10:  return "*"
  return ""

sig_df = pval_df.map(sig_stars)
print("\n── Significance (NW) ──")
print(sig_df)

检验beta是否稳定（滚动窗口回归）


In [ ]:
def rolling_ols(asset_series,orth_factors,window):
  T=len(asset_series)
  betas=np.full((T,orth_factors.shape[1]),np.nan)
  y_arr=asset_series.values
  X_arr=add_constant(orth_factors.values)

  for t in range(window-1,T):
    y_win=y_arr[t-window+1:t+1]
    X_win=X_arr[t-window+1:t+1]
    coef=np.linalg.lstsq(X_win,y_win,rcond=None)[0]
    betas[t]=coef[1:]

  return pd.DataFrame(betas,index=asset_series.index,columns=FACTOR_NAMES)


In [ ]:
WINDOW_36 = 36
WINDOW_60 = 60
rolling_betas_36={a: rolling_ols(df[a],orth_factors,WINDOW_36) for a in df.columns}
rolling_betas_60={a: rolling_ols(df[a],orth_factors,WINDOW_60) for a in df.columns}

检验beta系数在某一时刻前后是否发生变化

In [ ]:
def chow_test(y,X,break_idx):
  k=X.shape[1]

  def sse(y_,X_):
    r=y_-X_@np.linalg.lstsq(X_,y_,rcond=None)[0]
    return r@r

  SSE_r=sse(y,X)
  SSE_1=sse(y[:break_idx],X[:break_idx])
  SSE_2=sse(y[break_idx:],X[break_idx:])
  SSE_u=SSE_1+SSE_2

  n=len(y)
  F_stat=((SSE_r-SSE_u)/k)/(SSE_u/(n-2*k))
  p_val=1-stats.f.cdf(F_stat,dfn=k,dfd=n-2*k)
  return F_stat,p_val

In [ ]:
break_label="2008-09"
mask=df.index.strftime("%Y-%m")==break_label
if mask.any():
  break_idx=mask.argmax()
else:
  break_idx=T//2

X_full=add_constant(orth_factors.values,has_constant="add")
print(f"\n── Chow test (break at t={break_idx}, ~{df.index[break_idx].date()}) ──")
chow_results={}
for asset in df.columns:
  F,p=chow_test(df[asset].values,X_full,break_idx)
  chow_results[asset]={"F_stat":round(F,3),"p_value":round(p,4)}
chow_df = pd.DataFrame(chow_results).T
print(chow_df[chow_df["p_value"] < 0.10].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Plot A: Beta heatmap ──────────────────────────────────────
ax = axes[0]
im = ax.imshow(betas.values, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
ax.set_xticks(range(len(FACTOR_NAMES)))
ax.set_xticklabels(FACTOR_NAMES, fontsize=9)
ax.set_yticks(range(len(df.columns)))
ax.set_yticklabels(df.columns, fontsize=8)
ax.set_title("Full-sample betas (orth. factors)", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)

# Annotate with stars for significant betas
for i, asset in enumerate(df.columns):
    for j, fac in enumerate(FACTOR_NAMES):
        star = sig_stars(pval_df.loc[asset, fac])
        if star:
            ax.text(j, i, star, ha="center", va="center", fontsize=7, color="k")

# ── Plot B: Rolling betas for first 4 assets, Credit factor ────
ax2 = axes[1]
fac_to_plot = "Credit"
assets_to_plot = df.columns[:4]
colors = ["#2166ac", "#d6604d", "#4dac26", "#7b3294"]

for asset, color in zip(assets_to_plot, colors):
    r36 = rolling_betas_36[asset][fac_to_plot].dropna()
    r60 = rolling_betas_60[asset][fac_to_plot].dropna()
    ax2.plot(r36.index, r36.values, lw=1,   color=color, alpha=0.5, linestyle="--",
             label=f"{asset} 36m")
    ax2.plot(r60.index, r60.values, lw=1.5, color=color, alpha=0.9, linestyle="-",
             label=f"{asset} 60m")

ax2.axhline(0, color="gray", lw=0.8, linestyle=":")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator(4))
ax2.set_title(f"Rolling beta — {fac_to_plot} factor (dashed=36m, solid=60m)", fontsize=11)
ax2.set_xlabel("Date")
ax2.set_ylabel("Beta")
ax2.legend(fontsize=7, ncol=2)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


将上面的回归结果pack up给下一个模块使用

In [ ]:
results={
    "betas":betas,
    "alphas":alphas,
    "resid_var":resid_var,
    "tstat_df":tstat_df,
    "pval_df":pval_df,
    "r2":r2,
    "sig_df":sig_df,
    "orth_factors":orth_factors,
    "rolling_betas_36":rolling_betas_36,
    "rolling_betas_60":rolling_betas_60,
    "chow_df":chow_df,
    "nlags":nlags
}


模块3：得到收益率以及协方差矩阵（利用历史平均和因子方式）

In [ ]:
import matplotlib.ticker as mticker
from scipy.linalg import cholesky

In [ ]:
factor_premia_hist=raw_factors.mean()*12

ltcma_values={
    "Equities":0.067,
    "US Rates":0.04,
    "Risk Free Rate":0.035,
    "Credit":0.052,
    "StructCredit":0.061
}
factor_premia_ltcma=pd.Series(ltcma_values,name="LTCMA")
print(pd.DataFrame({
    "Historical": factor_premia_hist.round(4),
    "LTCMA":   factor_premia_ltcma.round(4),
}))

In [ ]:
mu_factor_hist=betas@factor_premia_hist
mu_factor_ltcma=betas@factor_premia_ltcma

mu_hist=df.mean()*12

In [ ]:
Sigma_sample=df.cov()*12

Sigma_F=orth_factors.cov()*12
B=betas.values
Sigma_e=np.diag(resid_var.values*12)
Sigma_factor_arr=B@Sigma_F.values@B.T+Sigma_e
Sigma_factor=pd.DataFrame(Sigma_factor_arr,index=df.columns,columns=df.columns)

In [ ]:
def check_psd(mat,name):
  eigmin=np.linalg.eigvals(mat).min()
  ok=bool(eigmin > 0)
  print(f"[PSD] {name}: min_eigval={eigmin:.2e}  "
      f"{'✓ positive definite' if ok else '✗ NOT positive definite'}")
  return ok

check_psd(Sigma_sample.values, "Sigma_sample")
check_psd(Sigma_factor_arr, "Sigma_factor")

模块4：构建最优组合（等权重和mean-variance)

In [ ]:
import cvxpy as cp

In [ ]:
def portfolio_stats(w,mu,Sigma):
  ret=float(w@mu.values)
  vol=float(np.sqrt(w@Sigma@w))
  sharpe=ret/vol if vol>1e-10 else np.nan
  return {"return":ret,"vol":vol,"sharpe":sharpe}

def run_mvo(mu,Sigma,lam,long_only=True,w_max=1.0):
  n=len(mu)
  w=cp.Variable(n)
  obj=cp.Maximize(mu.values@w-(lam/2)*cp.quad_form(w,Sigma.values))
  cons=[cp.sum(w)==1]
  if long_only:
    cons+=[w>=0]
  cons+=[w<=w_max]
  prob=cp.Problem(obj,cons)
  prob.solve(solver=cp.CLARABEL,verbose=False)
  if prob.status in ("optimal","optimal_inaccurate") and w.value is not None:
    weights=np.clip(w.value,0,None)
    weights/=weights.sum()
    return weights
  return None

def efficient_frontier(mu,Sigma,n_points=60,lam_min=0.1,lam_max=200):
  lambdas=np.logspace(np.log10(lam_min),np.log10(lam_max),n_points)
  rows=[]
  for lam in lambdas:
    w=run_mvo(mu,Sigma,lam)
    if w is not None:
      stats=portfolio_stats(w,mu,Sigma)
      row={"lambda":lam,**stats}
      row.update({a:w[i] for i,a in enumerate(mu.index)})
      rows.append(row)
  return pd.DataFrame(rows)

等权重asset class

In [ ]:
N_ASSETS=len(df.columns)
w_ew=np.ones(N_ASSETS)/N_ASSETS
ew_assets=pd.Series(w_ew,index=df.columns,name="EW")

ew_stats_sample=portfolio_stats(w_ew,mu_hist,Sigma_sample)
ew_stats_factor=portfolio_stats(w_ew,mu_factor_ltcma,Sigma_factor)

print("── Equal Weight 组合 ──")
print(f"  权重：{w_ew[0]:.4f} × {N_ASSETS} 个资产")
print(f"  [sample μ,Σ]  ret={ew_stats_sample['return']:.2%}  "
      f"vol={ew_stats_sample['vol']:.2%}  SR={ew_stats_sample['sharpe']:.3f}")
print(f"  [factor μ,Σ]  ret={ew_stats_factor['return']:.2%}  "
      f"vol={ew_stats_factor['vol']:.2%}  SR={ew_stats_factor['sharpe']:.3f}")

等权重因子组合

In [ ]:
from cvxpy import constraints
def solve_factor_ew(betas,long_only=True):
  N=betas.shape[0]
  K=betas.shape[1]
  B=betas.values

  w=cp.Variable(N)
  target_exp=np.ones(K)/K
  objective=cp.Minimize(cp.sum_squares(w))

  constraints=[B.T@w==target_exp,cp.sum(w)==1]

  if long_only:
    constraints+=[w>=0]

  prob=cp.Problem(objective,constraints)
  prob.solve(solver=cp.CLARABEL,verbose=False)

  if prob.status in ("optimal","optimal_inaccurate") and w.value is not None:
    weights=np.clip(w.value,0,None)
    weights/=weights.sum()
    return weights
  return None

w_factor_ew=solve_factor_ew(betas)
if w_factor_ew is None:
  print("[警告] 因子EW求解失败——可能是等权约束在long-only下无可行解")
  print("尝试放开做空约束...")
  w_factor_ew=solve_factor_ew(betas,long_only=False)

assert w_factor_ew is not None,"因子EW求解彻底失败，请检查beta矩阵"



In [ ]:
actual_exposure = betas.values.T @ w_factor_ew    # Bᵀw，shape (5,)
target_exposure = np.ones(len(FACTOR_NAMES)) / len(FACTOR_NAMES)

exposure_df = pd.DataFrame({
    "Target (1/K)": target_exposure.round(4),
    "Actual":        actual_exposure.round(4),
    "Diff":         (actual_exposure - target_exposure).round(6),
}, index=FACTOR_NAMES)

print("── 因子暴露验证 ──")
print(exposure_df)
max_dev = np.abs(actual_exposure - target_exposure).max()
print(f"最大偏差: {max_dev:.2e}  {'✓ OK' if max_dev < 1e-4 else '✗ CHECK'}")

In [ ]:
few_stats_sample = portfolio_stats(w_factor_ew, mu_hist,         Sigma_sample)
few_stats_factor = portfolio_stats(w_factor_ew, mu_factor_ltcma, Sigma_factor)

print("\n── 因子EW组合统计量 ──")
print(f"  [sample μ,Σ]  ret={few_stats_sample['return']:.2%}  "
      f"vol={few_stats_sample['vol']:.2%}  SR={few_stats_sample['sharpe']:.3f}")
print(f"  [factor μ,Σ]  ret={few_stats_factor['return']:.2%}  "
      f"vol={few_stats_factor['vol']:.2%}  SR={few_stats_factor['sharpe']:.3f}")

Mean-Variance Method(固定风险厌恶系数lambda)

In [ ]:
LAM_BASE=5.0
w_trad=run_mvo(mu_hist,Sigma_sample,LAM_BASE)
w_factor=run_mvo(mu_factor_ltcma,Sigma_factor,LAM_BASE)
w_factor_h=run_mvo(mu_factor_hist,Sigma_factor,LAM_BASE)
assert w_trad   is not None, "Traditional MVO failed"
assert w_factor is not None, "Factor MVO failed"

In [ ]:
trad_stats=portfolio_stats(w_trad,mu_hist,Sigma_sample)
factor_stats=portfolio_stats(w_factor,mu_factor_ltcma,Sigma_factor)
print(f"\n── MVO 单点组合（λ={LAM_BASE}） ──")
print(f"  Traditional SAA  ret={trad_stats['return']:.2%}  "
      f"vol={trad_stats['vol']:.2%}  SR={trad_stats['sharpe']:.3f}")
print(f"  Factor SAA       ret={factor_stats['return']:.2%}  "
      f"vol={factor_stats['vol']:.2%}  SR={factor_stats['sharpe']:.3f}")

# 权重对比表
weights_df = pd.DataFrame({
    "Equal Weight(asset class)":   w_ew,
    "Equal Weight(factor class)": w_factor_ew,
    "Traditional MVO": w_trad,
    "Factor MVO":     w_factor,
}, index=df.columns).round(4)
print("\n── 权重对比表 ──")
print(weights_df)

有效前沿（多个lambda)

In [ ]:
print("\n[Frontier] 扫描传统 SAA 有效前沿...")
frontier_trad   = efficient_frontier(mu_hist,         Sigma_sample)

print("[Frontier] 扫描因子 SAA 有效前沿...")
frontier_factor = efficient_frontier(mu_factor_ltcma, Sigma_factor)

print(f"  Traditional: {len(frontier_trad)} points  |  "
      f"Factor: {len(frontier_factor)} points")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor("#fafafa")

# ── Plot A: 权重条形图 ────────────────────────────────────────
ax1 = axes[0]
y   = np.arange(N_ASSETS)
h   = 0.20
c4  = ["#59a14f", "#e15759", "#4e79a7", "#f28e2b"]

ax1.barh(y + 1.5*h, w_ew,          h, label="Asset EW",        color=c4[0], alpha=0.85)
ax1.barh(y + 0.5*h, w_factor_ew,   h, label="Factor EW",       color=c4[1], alpha=0.85)
ax1.barh(y - 0.5*h, w_trad,        h, label="Traditional MVO", color=c4[2], alpha=0.85)
ax1.barh(y - 1.5*h, w_factor,      h, label="Factor MVO",      color=c4[3], alpha=0.85)

ax1.set_yticks(y)
ax1.set_yticklabels(df.columns, fontsize=8)
ax1.axvline(1/N_ASSETS, color="gray", lw=0.8, linestyle=":",
            label=f"1/N = {1/N_ASSETS:.3f}")
ax1.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax1.set_xlabel("Portfolio weight", fontsize=9)
ax1.set_title("Plot A — Portfolio weights:\nAsset EW vs Factor EW vs MVO",
              fontsize=10, fontweight="bold")
ax1.legend(fontsize=8)
ax1.grid(axis="x", alpha=0.3, lw=0.6)
ax1.set_facecolor("#f8f8f8")

# ── Plot B: 因子暴露对比（雷达图） ───────────────────────────
ax2 = axes[1]

# 四种组合各自的因子暴露 Bᵀw
exposures = {
    "Asset EW":        betas.values.T @ w_ew,
    "Factor EW":       betas.values.T @ w_factor_ew,
    "Traditional MVO": betas.values.T @ w_trad,
    "Factor MVO":      betas.values.T @ w_factor,
}

x    = np.arange(len(FACTOR_NAMES))
h2   = 0.18

for i, (label, exp) in enumerate(exposures.items()):
    offset = (i - 1.5) * h2
    ax2.bar(x + offset, exp, h2, label=label, color=c4[i], alpha=0.85)

ax2.axhline(1/len(FACTOR_NAMES), color="gray", lw=0.8, linestyle=":",
            label=f"Equal exposure (1/K={1/len(FACTOR_NAMES):.2f})")
ax2.set_xticks(x)
ax2.set_xticklabels(FACTOR_NAMES, fontsize=9)
ax2.set_ylabel("Factor exposure (Bᵀw)", fontsize=9)
ax2.set_title("Plot B — Factor exposures by portfolio:\nBᵀw for each construction method",
              fontsize=10, fontweight="bold")
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.3, lw=0.6)
ax2.set_facecolor("#f8f8f8")

plt.tight_layout(w_pad=3)
plt.show()
# ── Plot C: 将 Factor EW 追加到有效前沿图（重绘） ────────────
fig2 = plt.figure(figsize=(10, 7))
fig2.patch.set_facecolor("#fafafa")
ax_f = fig2.add_subplot(1, 1, 1)

# 复用已有的两条前沿曲线
ax_f.plot(frontier_trad["vol"],   frontier_trad["return"],
          lw=2, color="#4e79a7", label="Traditional SAA frontier")
ax_f.plot(frontier_factor["vol"], frontier_factor["return"],
          lw=2, color="#f28e2b", label="Factor SAA frontier")

# Asset EW 散点
ax_f.scatter(ew_stats_sample["vol"], ew_stats_sample["return"],
             marker="*", s=220, color="#4e79a7", zorder=5,
             label="Asset EW (sample μ,Σ)")
ax_f.scatter(ew_stats_factor["vol"], ew_stats_factor["return"],
             marker="*", s=220, color="#f28e2b", zorder=5,
             label="Asset EW (factor μ,Σ)")

# Factor EW 散点（新增）
ax_f.scatter(few_stats_factor["vol"], few_stats_factor["return"],
             marker="P", s=200, color="#e15759", zorder=5,
             label="Factor EW (factor μ,Σ)")
ax_f.scatter(few_stats_sample["vol"], few_stats_sample["return"],
             marker="P", s=200, color="#76b7b2", zorder=5,
             label="Factor EW (sample μ,Σ)")

# 代表性 MVO 点
ax_f.scatter(trad_stats["vol"],   trad_stats["return"],
             marker="D", s=80, color="#4e79a7", zorder=6)
ax_f.scatter(factor_stats["vol"], factor_stats["return"],
             marker="D", s=80, color="#f28e2b", zorder=6)
ax_f.annotate(f"λ={LAM_BASE}", (trad_stats["vol"],   trad_stats["return"]),
              textcoords="offset points", xytext=(6, 4), fontsize=8, color="#4e79a7")
ax_f.annotate(f"λ={LAM_BASE}", (factor_stats["vol"], factor_stats["return"]),
              textcoords="offset points", xytext=(6, 4), fontsize=8, color="#f28e2b")

ax_f.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax_f.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax_f.set_xlabel("Annualised volatility", fontsize=10)
ax_f.set_ylabel("Annualised return",     fontsize=10)
ax_f.set_title("Efficient frontier — Traditional vs Factor SAA\n"
               "(★ = Asset EW,  ✚ = Factor EW,  ◆ = MVO at λ=5)",
               fontsize=11, fontweight="bold")
ax_f.legend(fontsize=8.5)
ax_f.grid(alpha=0.3, lw=0.6)
ax_f.set_facecolor("#f8f8f8")

plt.tight_layout()
plt.show()

模块5：策略表现以及滚动表现


In [ ]:
IS_FRAC=0.7
IS_END=int(T*IS_FRAC)
dates_is=df.index[:IS_END]
dates_oos=df.index[IS_END:]
df_is=df.iloc[:IS_END]
df_oos=df.iloc[IS_END:]

print(f"In-Sample  : {dates_is[0].date()} → {dates_is[-1].date()}  ({IS_END} months)")
print(f"Out-of-Sample: {dates_oos[0].date()} → {dates_oos[-1].date()}  ({len(dates_oos)} months)")

In [ ]:
def estimate_weights_on_window(df_win,raw_factors_win,lam=5):
  n=len(df_win)
  orth_win=orthogonalize_factors(raw_factors_win)

  betas_w,resid_var_w=[],[]
  for asset in df_win.columns:
    y_=df_win[asset].values
    X_=add_constant(orth_win.values,has_constant="add")
    m=OLS(y_,X_).fit()
    betas_w.append(m.params[1:])
    resid_var_w.append(m.mse_resid)
  B_w=np.array(betas_w)
  rv_w=np.array(resid_var_w)

  mu_s=df_win.mean()*12
  Sig_s=df_win.cov()*12

  mu_f=pd.Series(B_w@factor_premia_ltcma.values,index=df_win.columns)
  Sig_F=orth_win.cov()*12
  Sig_e=np.diag(rv_w*12)
  Sig_f=pd.DataFrame(B_w@Sig_F.values@B_w.T+Sig_e,index=df_win.columns,columns=df_win.columns)

  w_t=run_mvo(mu_s,Sig_s,lam)
  w_fm=run_mvo(mu_f,Sig_f,lam)

  N=df_win.shape[1]
  w_a=np.ones(N)/N

  B_df=pd.DataFrame(B_w,index=df_win.columns,columns=raw_factors_win.columns)
  w_fe=solve_factor_ew(B_df)
  if w_fe is None:
    w_fe=w_a

  return {"trad":w_t,"factor":w_fm,"asset_ew":w_a,"factor_ew":w_fe}

w_is=estimate_weights_on_window(df_is,raw_factors.iloc[:IS_END])

In [ ]:
def compute_nav(returns_oos,weights):
  port_ret=returns_oos.values@weights
  nav=pd.Series(np.cumprod(1+port_ret),index=returns_oos.index,name="NAV")
  return nav

def performance_metrics(nav,freq=12):
  ret=nav.pct_change().dropna()
  T_=len(ret)
  cagr=nav.iloc[-1]**(freq/T)-1
  vol=ret.std()*np.sqrt(freq)
  sharpe=cagr/vol if vol>1e-10 else np.nan

  roll_max=nav.cummax()
  drawdown=(nav-roll_max)/roll_max
  max_dd=drawdown.min()

  calmar=cagr/abs(max_dd) if max_dd<0 else np.nan
  return{
      "CAGR":round(cagr,4),
      "Vol":round(vol,4),
      "Sharpe":round(sharpe,4),
      "Max_DD":round(max_dd,4),
      "Calmar":round(calmar,4)
  }


portfolios={
    "Asset EW":w_is["asset_ew"],
    "Factor EW":w_is["factor_ew"],
    "Traditional MVO":w_is["trad"],
    "Factor MVO":w_is["factor"]
}

nav_dict={}
metrics_rows={}
for name,w in portfolios.items():
  if w is None:
    print(f"[跳过]{name}:权重为None")
    continue
  nav_dict[name]=compute_nav(df_oos,w)
  metrics_rows[name]=performance_metrics(nav_dict[name])

metrics_df=pd.DataFrame(metrics_rows).T
print(metrics_df.to_string())

In [ ]:
B_full=betas.values
exposure_rows={}
for name,w in portfolios.items():
  if w is None:
    continue
  exp=B_full.T@w
  exposure_rows[name]=dict(zip(FACTOR_NAMES,exp.round(4)))
exposure_port=pd.DataFrame(exposure_rows).T
print("\n── 组合因子暴露 wᵀB ──")
print(exposure_port)

滚动回测

In [ ]:
TRAIN_WIN=60
REBAL_FREQ=12
def walk_forward(df_all,raw_factors_all,train_win,rebal_freq,lam=5.0):
  T_all=len(df_all)
  starts=range(train_win,T_all,rebal_freq)
  port_names=["Asset EW","Factor EW","Traditional MVO","Factor MVO"]
  monthly_ret={p:[] for p in port_names}
  monthly_idx=[]

  for start in starts:
    end=min(start+rebal_freq,T_all)
    df_train=df_all.iloc[start-train_win:start]
    fac_train=raw_factors_all.iloc[start-train_win:start]
    w_dict=estimate_weights_on_window(df_train,fac_train,lam)

    df_hold=df_all.iloc[start:end]
    for date_idx in range(len(df_hold)):
      row_ret=df_hold.iloc[date_idx].values
      for pname,wkey in zip(port_names,["asset_ew","factor_ew","trad","factor"]):
        w_=w_dict.get(wkey)
        if w_ is not None:
          monthly_ret[pname].append(row_ret@w_)
        else:
          monthly_ret[pname].append(np.nan)
      monthly_idx.append(df_hold.index[date_idx])
  idx_series=pd.DatetimeIndex(monthly_idx)
  result={}
  for p in port_names:
    result[p]=pd.Series(monthly_ret[p],index=idx_series,name=p)
  return result

wf_returns=walk_forward(df,raw_factors,TRAIN_WIN,REBAL_FREQ)
wf_nav={p:(1+wf_returns[p].dropna()).cumprod() for p in wf_returns}
wf_metrics={}
for p,nav_s in wf_nav.items():
  wf_metrics[p]=performance_metrics(nav_s)
wf_metrics_df=pd.DataFrame(wf_metrics).T
print(wf_metrics_df.to_string())

In [ ]:
COLORS = {
    "Asset EW":        "#59a14f",
    "Factor EW":       "#e15759",
    "Traditional MVO": "#4e79a7",
    "Factor MVO":      "#f28e2b",
}

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("#fafafa")

# ── Plot A: OOS 净值曲线 ──────────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
for name, nav in nav_dict.items():
    ax1.plot(nav.index, nav.values, lw=2,
             color=COLORS[name], label=name)
ax1.axhline(1, color="gray", lw=0.8, linestyle=":")
ax1.set_title("Plot A — OOS Cumulative NAV (fixed weights from IS)",
              fontsize=10, fontweight="bold")
ax1.set_ylabel("NAV (start = 1)", fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator(2))
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3, lw=0.6)
ax1.set_facecolor("#f8f8f8")

# ── Plot B: 绩效指标条形图 ────────────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
metrics_plot = metrics_df[["CAGR", "Vol", "Sharpe", "Max_DD"]].copy()
x   = np.arange(len(metrics_plot.columns))
h_b = 0.18
names_list = list(metrics_plot.index)

for i, name in enumerate(names_list):
    offset = (i - (len(names_list)-1)/2) * h_b
    vals   = metrics_plot.loc[name].values.astype(float)
    bars   = ax2.bar(x + offset, vals, h_b,
                     label=name, color=COLORS[name], alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels(["CAGR", "Volatility", "Sharpe", "Max Drawdown"],
                     fontsize=9)
ax2.axhline(0, color="gray", lw=0.8)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax2.set_title("Plot B — OOS Performance metrics comparison",
              fontsize=10, fontweight="bold")
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.3, lw=0.6)
ax2.set_facecolor("#f8f8f8")

# ── Plot C: 因子暴露热力图 ────────────────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
exp_mat = exposure_port.values.astype(float)
im = ax3.imshow(exp_mat, cmap="RdBu_r", aspect="auto",
                vmin=-0.5, vmax=max(1.0, exp_mat.max()))
ax3.set_xticks(range(len(FACTOR_NAMES)))
ax3.set_xticklabels(FACTOR_NAMES, fontsize=9)
ax3.set_yticks(range(len(exposure_port)))
ax3.set_yticklabels(exposure_port.index, fontsize=9)
ax3.set_title("Plot C — Portfolio factor exposures wᵀB",
              fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax3, shrink=0.8, label="Exposure")

# 数值标注
for i in range(exp_mat.shape[0]):
    for j in range(exp_mat.shape[1]):
        ax3.text(j, i, f"{exp_mat[i,j]:.2f}",
                 ha="center", va="center", fontsize=8,
                 color="white" if abs(exp_mat[i,j]) > 0.6 else "black")

for x_ in np.arange(-0.5, len(FACTOR_NAMES), 1):
    ax3.axvline(x_, color="white", lw=0.8)
for y_ in np.arange(-0.5, len(exposure_port), 1):
    ax3.axhline(y_, color="white", lw=0.8)

# ── Plot D: Walk-Forward 净值曲线 ─────────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
for name, nav_s in wf_nav.items():
    ax4.plot(nav_s.index, nav_s.values, lw=2,
             color=COLORS[name], label=name)
ax4.axhline(1, color="gray", lw=0.8, linestyle=":")
ax4.set_title(f"Plot D — Walk-Forward NAV\n"
              f"(train={TRAIN_WIN}m, rebal every {REBAL_FREQ}m)",
              fontsize=10, fontweight="bold")
ax4.set_ylabel("NAV (start = 1)", fontsize=9)
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax4.xaxis.set_major_locator(mdates.YearLocator(2))
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, lw=0.6)
ax4.set_facecolor("#f8f8f8")

plt.tight_layout(h_pad=3, w_pad=2)
plt.show()
